# RAG Pipeline on OpenShift AI — Setup, Run & Query Guide

This notebook walks you through the complete **Retrieval-Augmented Generation (RAG)** pipeline on **Red Hat OpenShift AI (RHOAI)**.

## Pipeline Overview

The pipeline has **4 steps**, orchestrated by Kubeflow Pipelines (KFP):

| Step | Component | Description |
|------|-----------|-------------|
| 1 | **Parse & Chunk** | RayJob distributes PDF parsing via Docling + HybridChunker, writes JSONL to S3 (MinIO) |
| 2 | **Ingest to Milvus** | Reads chunks from S3, embeds with sentence-transformers, inserts into Milvus |
| 3 | **Download Model** | Downloads LLM weights to a PVC (cached — skips if already present) |
| 4 | **Deploy Model** | Creates a vLLM InferenceService from the cached PVC model |

```
  Input PDFs (PVC)              HuggingFace
       │                             │
       ▼                             ▼
  ┌──────────┐                ┌──────────────┐
  │ 1. Parse │                │ 3. Download  │
  │ & Chunk  │                │    Model     │
  │ (RayJob) │                │  (to PVC)    │
  └────┬─────┘                └──────┬───────┘
       │ JSONL → S3                  │
       ▼                             ▼
  ┌──────────┐                ┌──────────────┐
  │ 2. Ingest│                │ 4. Deploy    │
  │ to Milvus│                │    Model     │
  │ (embed)  │                │  (vLLM ISVC) │
  └────┬─────┘                └──────┬───────┘
       │                             │
       ▼                             ▼
    Milvus DB ◄──── RAG Query ────► vLLM API
```

## What This Notebook Covers

1. **Prerequisites** — Create PVCs, Secrets, RBAC
2. **Compile the Pipeline** — Generate the KFP YAML
3. **Upload & Run** — Submit to Data Science Pipelines
4. **Monitor** — Track pipeline and component progress
5. **Test RAG** — Query the deployed model using Milvus-retrieved context

---
## 1. Prerequisites

Before running the pipeline, ensure the following are in place:

- **RHOAI** installed with Data Science Pipelines (DSPA) configured
- **KubeRay** operator enabled
- **Milvus** deployed (e.g., via Milvus Operator)
- **MinIO** or S3-compatible storage for intermediate chunks
- **GPU nodes** with NVIDIA GPUs for model serving

In [ ]:
# Install required packages
!pip install -q kfp kfp-kubernetes pymilvus sentence-transformers openai requests

In [ ]:
# ============================================================
# Configuration — Update these values for your environment
# ============================================================

NAMESPACE = "ray-docling"

# S3 / MinIO
S3_ENDPOINT = "http://minio-service.default.svc.cluster.local:9000"
S3_BUCKET = "rag-chunks"
S3_PREFIX = "chunks"
S3_SECRET_NAME = "minio-secret"  # K8s Secret with keys: access_key, secret_key

# Milvus
MILVUS_HOST = "milvus-milvus.milvus.svc.cluster.local"
MILVUS_PORT = 19530
COLLECTION_NAME = "rag_documents"

# Embedding
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384

# LLM
LLM_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
LLM_SERVED_NAME = "mistral-7b-instruct-v0-3"  # As registered in vLLM
MODEL_CACHE_PVC = "model-cache-pvc"

# Inference endpoint (set after pipeline runs, or if model is already deployed)
INFERENCE_URL = f"http://{LLM_SERVED_NAME}-predictor.{NAMESPACE}.svc.cluster.local/v1"

print(f"Namespace:      {NAMESPACE}")
print(f"S3 endpoint:    {S3_ENDPOINT}")
print(f"Milvus:         {MILVUS_HOST}:{MILVUS_PORT}")
print(f"LLM endpoint:   {INFERENCE_URL}")

### 1.1 Create Prerequisites (run once)

Apply the required Kubernetes manifests. Skip if already created.

In [ ]:
# Create the S3 credentials Secret
# Update access_key / secret_key below to match your MinIO credentials
!oc create secret generic minio-secret \
    --from-literal=access_key=minioadmin \
    --from-literal=secret_key=minioadmin \
    -n {NAMESPACE} --dry-run=client -o yaml | oc apply -f -

In [ ]:
# Create the model cache PVC (30Gi for Mistral-7B weights)
!oc apply -f manifests/model-cache-pvc.yaml

In [ ]:
# Create RBAC for the pipeline service account
!oc apply -f manifests/rbac.yaml

In [ ]:
# Verify all prerequisites
print("=== PVCs ===")
!oc get pvc -n {NAMESPACE}
print("\n=== Secrets ===")
!oc get secret {S3_SECRET_NAME} -n {NAMESPACE}
print("\n=== RBAC ===")
!oc get role pipeline-rag-role -n {NAMESPACE}
print("\n=== Pipeline Service Account ===")
!oc get sa -n {NAMESPACE} | grep -E 'pipeline|dspa'

---
## 2. Compile the Pipeline

The pipeline is defined in `pipeline_multistep.py` with 4 reusable KFP components.
Compiling generates a self-contained YAML that can be uploaded to Data Science Pipelines.

In [ ]:
!pip install -q codeflare-sdk kubernetes
!python pipeline_multistep.py

In [ ]:
import os
yaml_path = "rag_multistep_pipeline.yaml"
size_kb = os.path.getsize(yaml_path) / 1024
print(f"Compiled: {yaml_path} ({size_kb:.1f} KB)")

---
## 3. Upload & Run the Pipeline

You can upload the pipeline YAML to Data Science Pipelines via:

**Option A — RHOAI Dashboard (recommended):**
1. Open the RHOAI dashboard
2. Go to **Data Science Pipelines** → **Pipelines**
3. Click **Import pipeline** → upload `rag_multistep_pipeline.yaml`
4. Click **Create run**, review the parameters, and start

**Option B — KFP SDK (programmatic):**

In [ ]:
# Uncomment and configure to upload via KFP SDK
# import kfp
#
# # Get the DSPA route
# dspa_route = !oc get route ds-pipeline-dspa -n {NAMESPACE} -o jsonpath='{{.spec.host}}'
# dspa_url = f"https://{dspa_route[0]}"
#
# # Get auth token
# token = !oc whoami -t
#
# client = kfp.Client(
#     host=dspa_url,
#     existing_token=token[0],
#     ssl_ca_cert='/var/run/secrets/kubernetes.io/serviceaccount/ca.crt',  # in-cluster
# )
#
# # Upload pipeline
# pipeline = client.upload_pipeline(
#     pipeline_package_path="rag_multistep_pipeline.yaml",
#     pipeline_name="RAG Multi-Step Pipeline",
# )
# print(f"Pipeline uploaded: {pipeline.id}")
#
# # Create a run
# run = client.create_run_from_pipeline_package(
#     pipeline_file="rag_multistep_pipeline.yaml",
#     run_name="rag-test-run",
#     arguments={
#         "namespace": NAMESPACE,
#         "s3_endpoint": S3_ENDPOINT,
#         "s3_secret_name": S3_SECRET_NAME,
#         "milvus_host": MILVUS_HOST,
#         "llm_model_name": LLM_MODEL_NAME,
#     },
# )
# print(f"Run started: {run.run_id}")

---
## 4. Monitor Pipeline Execution

Use these cells to check the status of each pipeline step.

In [ ]:
# Step 1: Check RayJob status (Parse & Chunk)
print("=== RayJobs ===")
!oc get rayjob -n {NAMESPACE}
print("\n=== Ray Pods ===")
!oc get pods -n {NAMESPACE} -l ray.io/node-type

In [ ]:
# Step 2: Check pipeline pod logs (Ingest to Milvus)
!oc get pods -n {NAMESPACE} -l pipelines.kubeflow.org/v2_component=true --sort-by=.metadata.creationTimestamp

In [ ]:
# Steps 3 & 4: Check model deployment
print("=== InferenceService ===")
!oc get inferenceservice -n {NAMESPACE}
print("\n=== Model Serving Pod ===")
!oc get pods -n {NAMESPACE} -l serving.kserve.io/inferenceservice={LLM_SERVED_NAME}

---
## 5. Test the RAG Pipeline

Once all 4 pipeline steps complete, the RAG system is ready to query:
- **Milvus** contains embedded document chunks
- **vLLM** serves the Mistral-7B model

The RAG flow:
1. Embed the user's question using the same embedding model
2. Search Milvus for the most similar document chunks
3. Build a prompt with the retrieved context
4. Send to the LLM for answer generation

### 5.1 Verify Milvus Collection

Check that the ingestion step populated the vector database.

In [ ]:
from pymilvus import MilvusClient

# For in-cluster notebooks, use the service directly.
# For local notebooks, port-forward first:
#   oc port-forward svc/milvus-milvus 19530:19530 -n milvus
#   and set MILVUS_HOST = "localhost"

milvus_uri = f"http://{MILVUS_HOST}:{MILVUS_PORT}"
milvus_client = MilvusClient(uri=milvus_uri, db_name="default")

if not milvus_client.has_collection(COLLECTION_NAME):
    print(f"Collection '{COLLECTION_NAME}' not found.")
    print("The ingestion step may not have completed yet.")
else:
    milvus_client.load_collection(COLLECTION_NAME)
    stats = milvus_client.get_collection_stats(COLLECTION_NAME)
    row_count = stats.get("row_count", 0)
    print(f"Collection: {COLLECTION_NAME}")
    print(f"Vectors:    {row_count}")
    print(f"Stats:      {stats}")

In [ ]:
# Sample a few chunks to see what's in the collection
samples = milvus_client.query(
    collection_name=COLLECTION_NAME,
    filter="chunk_index >= 0",
    output_fields=["source_file", "chunk_index", "text"],
    limit=5,
)

print(f"Sample chunks ({len(samples)}):\n")
for s in samples:
    print(f"  [{s['source_file']} | chunk {s['chunk_index']}]")
    print(f"  {s['text'][:200]}...")
    print()

### 5.2 Verify Model Endpoint

Check that the vLLM model is serving and responding.

In [ ]:
import requests

print(f"Checking model endpoint: {INFERENCE_URL}")

try:
    resp = requests.get(f"{INFERENCE_URL}/models", timeout=10)
    resp.raise_for_status()
    models = resp.json()
    print(f"\nAvailable models:")
    for m in models.get("data", []):
        print(f"  - {m['id']}")
except requests.ConnectionError:
    print("\nConnection failed. The model may not be deployed yet.")
    print("Check: oc get inferenceservice -n", NAMESPACE)
except Exception as e:
    print(f"\nError: {e}")

### 5.3 RAG Query — Search & Ask

This is the core RAG loop:
1. **Embed** the question with the same model used during ingestion
2. **Search** Milvus for the top-K most similar chunks
3. **Build** a prompt with the retrieved context
4. **Generate** an answer using the LLM

In [ ]:
from openai import OpenAI
from sentence_transformers import SentenceTransformer

# Load the embedding model (same one used during ingestion)
embed_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Embedding model loaded: {EMBEDDING_MODEL} (dim={EMBEDDING_DIM})")

# OpenAI-compatible client pointing to vLLM
llm_client = OpenAI(base_url=INFERENCE_URL, api_key="unused")
print(f"LLM client configured: {INFERENCE_URL}")

In [ ]:
def search_similar_chunks(question, top_k=5):
    """Embed the question and search Milvus for similar chunks."""
    query_embedding = embed_model.encode(
        [question], normalize_embeddings=True
    ).tolist()

    results = milvus_client.search(
        collection_name=COLLECTION_NAME,
        data=query_embedding,
        limit=top_k,
        output_fields=["source_file", "chunk_index", "text"],
        search_params={"metric_type": "COSINE", "params": {"nprobe": 16}},
    )

    chunks = []
    for hits in results:
        for hit in hits:
            chunks.append({
                "text": hit["entity"]["text"],
                "source_file": hit["entity"]["source_file"],
                "chunk_index": hit["entity"]["chunk_index"],
                "score": hit["distance"],
            })
    return chunks


def build_rag_prompt(question, chunks):
    """Build a prompt with retrieved context for the LLM."""
    context_block = "\n\n---\n\n".join(
        f"[Source: {c['source_file']}, chunk {c['chunk_index']}, "
        f"similarity: {c['score']:.3f}]\n{c['text']}"
        for c in chunks
    )
    return (
        "You are a helpful assistant. Answer the user's question based on the "
        "provided context documents. If the answer is not in the context, say so.\n\n"
        f"## Context\n\n{context_block}\n\n"
        f"## Question\n\n{question}\n\n"
        "## Answer\n\n"
    )


def ask_llm(prompt, max_tokens=512, temperature=0.1):
    """Send the prompt to vLLM and return the response."""
    response = llm_client.chat.completions.create(
        model=LLM_SERVED_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return response.choices[0].message.content


def rag_query(question, top_k=5, max_tokens=512, temperature=0.1):
    """End-to-end RAG: retrieve context → build prompt → generate answer."""
    # Step 1: Retrieve similar chunks
    chunks = search_similar_chunks(question, top_k=top_k)

    # Step 2: Build prompt with context
    prompt = build_rag_prompt(question, chunks)

    # Step 3: Generate answer
    answer = ask_llm(prompt, max_tokens=max_tokens, temperature=temperature)

    return {
        "question": question,
        "answer": answer,
        "sources": chunks,
        "prompt": prompt,
    }


print("RAG functions ready.")

### 5.4 Try a RAG Query

Ask a question about your ingested documents. The system will:
1. Find the most relevant chunks in Milvus
2. Send them as context to Mistral-7B
3. Return a grounded answer with source references

In [ ]:
question = "What are the main topics covered in the documents?"

result = rag_query(question)

print(f"Question: {result['question']}\n")
print(f"Answer:\n{result['answer']}\n")
print(f"Sources ({len(result['sources'])}):\n")
for s in result["sources"]:
    print(f"  - {s['source_file']} (chunk {s['chunk_index']}, similarity: {s['score']:.3f})")

In [ ]:
# Try another question — replace with something relevant to your documents
result = rag_query("Summarize the key findings.")

print(f"Question: {result['question']}\n")
print(f"Answer:\n{result['answer']}\n")
print(f"Sources ({len(result['sources'])}):\n")
for s in result["sources"]:
    print(f"  - {s['source_file']} (chunk {s['chunk_index']}, similarity: {s['score']:.3f})")

### 5.5 Inspect the RAG Process

You can inspect each step independently — search for chunks without sending to the LLM,
or view the full prompt that was sent.

In [ ]:
# Search only — view retrieved chunks without querying the LLM
question = "What are the main topics covered in the documents?"
chunks = search_similar_chunks(question, top_k=5)

print(f"Top {len(chunks)} chunks for: '{question}'\n")
for i, c in enumerate(chunks, 1):
    print(f"--- Chunk {i} (score: {c['score']:.3f}) ---")
    print(f"Source: {c['source_file']}, chunk_index: {c['chunk_index']}")
    print(c["text"][:500])
    print()

In [ ]:
# View the full prompt sent to the LLM
prompt = build_rag_prompt(question, chunks)
print(f"Prompt length: {len(prompt)} characters\n")
print(prompt)

In [ ]:
# Query the LLM directly (without RAG context) for comparison
direct_response = llm_client.chat.completions.create(
    model=LLM_SERVED_NAME,
    messages=[{"role": "user", "content": question}],
    max_tokens=512,
    temperature=0.1,
)

print("LLM response WITHOUT context (no RAG):\n")
print(direct_response.choices[0].message.content)
print("\n" + "=" * 60)
print("\nNotice how the answer without RAG context is generic,")
print("while the RAG answer references specific document content.")

### 5.6 Interactive Query Loop

Run this cell to enter an interactive Q&A session.
Type your questions and get RAG-powered answers. Type `quit` to exit.

In [ ]:
while True:
    question = input("\nAsk a question (or 'quit'): ").strip()
    if question.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    if not question:
        continue

    result = rag_query(question)
    print(f"\nAnswer:\n{result['answer']}\n")
    print(f"Sources:")
    for s in result["sources"]:
        print(f"  - {s['source_file']} (chunk {s['chunk_index']}, {s['score']:.3f})")

---
## 6. Cleanup

Uncomment and run the cells below to tear down resources.

In [ ]:
# Delete the InferenceService (stops the model serving pod)
# !oc delete inferenceservice {LLM_SERVED_NAME} -n {NAMESPACE}

# Delete the ServingRuntime
# !oc delete servingruntime {LLM_SERVED_NAME} -n {NAMESPACE}

# Drop the Milvus collection
# milvus_client.drop_collection(COLLECTION_NAME)
# print(f"Dropped collection: {COLLECTION_NAME}")

# Delete the model cache PVC (removes downloaded model weights)
# !oc delete pvc {MODEL_CACHE_PVC} -n {NAMESPACE}

# Delete RBAC
# !oc delete -f manifests/rbac.yaml